[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/07_ONNX_Runtime/03_Inference_Sessions/Inference_Sessions_Deep_Dive.ipynb)

# Inference Sessions — Deep Dive

A comprehensive exploration of ONNX Runtime's `InferenceSession` API: `SessionOptions` configuration,
`RunOptions` per-request controls, `IOBinding` for zero-copy inference, memory pattern optimization,
batch inference strategies, warm-up effects, concurrent sessions, and production profiling.

---

## Table of Contents

| # | Section | Focus |
|---|---------|-------|
| 1 | [SessionOptions Deep Dive](#1-sessionoptions-deep-dive) | Threading, optimization levels, execution modes |
| 2 | [RunOptions](#2-runoptions) | Per-invocation controls and cancellation |
| 3 | [IOBinding Mechanism](#3-iobinding-mechanism) | Zero-copy data flow and device buffers |
| 4 | [Memory Patterns](#4-memory-patterns) | Arena allocation and buffer reuse |
| 5 | [Batch Inference Strategies](#5-batch-inference-strategies) | Static, dynamic, micro-batching |
| 6 | [Warm-up Effects](#6-warm-up-effects) | First-run costs and JIT compilation |
| 7 | [Concurrent Sessions](#7-concurrent-sessions) | Multi-model serving and thread budgets |
| 8 | [Model Metadata](#8-model-metadata) | Inspecting I/O schemas and model properties |
| 9 | [Serialization](#9-serialization) | Saving optimized graphs and session state |
| 10 | [Profiling](#10-profiling) | Built-in profiler and performance analysis |

In [ ]:
# Install dependencies
!pip install onnxruntime onnx numpy matplotlib -q

<a id='1'></a>
## 1. Session Fundamentals

The `InferenceSession` is ORT's central abstraction — it encapsulates a **fully optimized, partitioned, and memory-planned** computation graph ready for repeated execution.

### Lifecycle Overview

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                    InferenceSession Lifecycle                                 │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│  Phase 1: CONSTRUCTION                                                       │
│  ┌──────────┐   ┌──────────────┐   ┌────────────┐   ┌───────────────┐      │
│  │  Load    │──►│  Optimize    │──►│  Partition  │──►│  Plan Memory  │      │
│  │  Model   │   │  Graph       │   │  to EPs     │   │  & Execution  │      │
│  └──────────┘   └──────────────┘   └────────────┘   └───────────────┘      │
│       │                                                      │               │
│       └──────────── EXPENSIVE (one-time cost) ───────────────┘               │
│                                                                              │
│  Phase 2: EXECUTION (repeated)                                               │
│  ┌──────────┐   ┌──────────────┐   ┌────────────┐   ┌───────────────┐      │
│  │  Bind    │──►│  Execute     │──►│  Collect   │──►│  Return       │      │
│  │  Inputs  │   │  Kernels     │   │  Outputs   │   │  OrtValues    │      │
│  └──────────┘   └──────────────┘   └────────────┘   └───────────────┘      │
│       │                                                      │               │
│       └──────────── CHEAP (per-request cost) ────────────────┘               │
│                                                                              │
└─────────────────────────────────────────────────────────────────────────────┘
```

### Cost Model

The total cost of serving $N$ inference requests:

$$C_{\text{total}} = C_{\text{create}} + N \cdot C_{\text{run}}$$

Where:
- $C_{\text{create}} = T_{\text{load}} + T_{\text{optimize}} + T_{\text{partition}} + T_{\text{plan}}$
- $C_{\text{run}} = T_{\text{copy}} + T_{\text{compute}} + T_{\text{sync}}$

The **amortized cost per request** converges to the run cost:

$$\lim_{N \to \infty} \frac{C_{\text{total}}}{N} = C_{\text{run}}$$

This justifies investing heavily in session creation optimizations (higher `graph_optimization_level`) because the one-time cost is amortized across millions of requests.

### Session as a Compiled Artifact

Think of `InferenceSession` as an analogy to compiled code:

| Source Code Analogy | ONNX Runtime |
|--------------------|--------------|
| `.c` source file | `.onnx` model file |
| Compiler (`gcc -O3`) | Graph optimizer + partitioner |
| Compiled binary | Optimized execution plan |
| Process execution | `session.run()` |
| Linking libraries | Loading Execution Providers |

In [ ]:
import onnxruntime as ort
import onnx
from onnx import helper, TensorProto, numpy_helper
import numpy as np
import time

print(f"ONNX Runtime version: {ort.__version__}")
print(f"Available providers: {ort.get_available_providers()}")

# Build a reference model for demonstrations
np.random.seed(42)
W1 = np.random.randn(784, 256).astype(np.float32) * 0.01
B1 = np.zeros(256, dtype=np.float32)
W2 = np.random.randn(256, 128).astype(np.float32) * 0.01
B2 = np.zeros(128, dtype=np.float32)
W3 = np.random.randn(128, 10).astype(np.float32) * 0.01
B3 = np.zeros(10, dtype=np.float32)

X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", 784])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["batch", 10])

nodes = [
    helper.make_node("MatMul", ["X", "W1"], ["H1"]),
    helper.make_node("Add", ["H1", "B1"], ["H1b"]),
    helper.make_node("Relu", ["H1b"], ["H1r"]),
    helper.make_node("MatMul", ["H1r", "W2"], ["H2"]),
    helper.make_node("Add", ["H2", "B2"], ["H2b"]),
    helper.make_node("Relu", ["H2b"], ["H2r"]),
    helper.make_node("MatMul", ["H2r", "W3"], ["H3"]),
    helper.make_node("Add", ["H3", "B3"], ["H3b"]),
    helper.make_node("Softmax", ["H3b"], ["Y"], axis=1),
]

graph = helper.make_graph(
    nodes, "MLP_3Layer", [X], [Y],
    initializer=[
        numpy_helper.from_array(W1, "W1"), numpy_helper.from_array(B1, "B1"),
        numpy_helper.from_array(W2, "W2"), numpy_helper.from_array(B2, "B2"),
        numpy_helper.from_array(W3, "W3"), numpy_helper.from_array(B3, "B3"),
    ]
)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
onnx.checker.check_model(model)
onnx.save(model, "session_demo.onnx")
print(f"\nDemo model: 3-layer MLP (784→256→128→10), {len(nodes)} nodes")
print(f"Parameters: {sum(p.size for p in [W1,B1,W2,B2,W3,B3]):,}")

<a id='2'></a>
## 2. SessionOptions — Configuration Space

`SessionOptions` controls **how** the session is constructed. These settings are frozen at creation time and cannot be changed for an existing session.

### Configuration Taxonomy

```
SessionOptions
├── Threading
│   ├── intra_op_num_threads    (parallelism within one operator)
│   └── inter_op_num_threads    (parallelism across operators)
├── Graph Optimization
│   ├── graph_optimization_level (DISABLE → BASIC → EXTENDED → ALL)
│   └── optimized_model_filepath (save optimized graph for inspection)
├── Execution
│   ├── execution_mode          (SEQUENTIAL vs PARALLEL)
│   └── execution_order         (topological ordering variant)
├── Memory
│   ├── enable_mem_pattern       (reuse allocation pattern)
│   ├── enable_cpu_mem_arena     (arena allocator)
│   └── enable_mem_reuse         (buffer sharing)
├── Providers
│   └── add_session_config_entry (EP-specific knobs)
└── Debugging
    ├── enable_profiling         (chrome trace output)
    ├── log_severity_level       (verbosity)
    └── log_verbosity_level      (detail level)
```

### Optimization Levels Formal Definition

Each level is a superset of the previous:

$$\text{DISABLE} \subset \text{BASIC} \subset \text{EXTENDED} \subset \text{ALL}$$

| Level | Passes Applied | Typical Speedup |
|-------|---------------|----------------|
| DISABLE | None | 1.0x (baseline) |
| BASIC | Constant folding, dead code elimination, redundant node removal | 1.1-1.3x |
| EXTENDED | Operator fusion (Conv+BN+Relu, MatMul+Add), layout transforms | 1.3-2.0x |
| ALL | All extended + NHWC layout optimization, attention fusion | 1.5-3.0x |

### Threading Model

The relationship between intra-op and inter-op threads:

$$\text{Total threads} = \text{inter\_op} \times \text{intra\_op}$$

**Constraint**: Total threads should not exceed physical cores to avoid contention:

$$\text{inter\_op} \times \text{intra\_op} \leq N_{\text{physical\_cores}}$$

For sequential CNN/Transformer inference (no graph-level parallelism):
- Set `inter_op_num_threads = 1`
- Set `intra_op_num_threads = N_{\text{physical\_cores}}`

In [ ]:
import onnxruntime as ort
import time
import numpy as np

# Demonstrate SessionOptions impact on creation and execution
levels = {
    "DISABLE": ort.GraphOptimizationLevel.ORT_DISABLE_ALL,
    "BASIC": ort.GraphOptimizationLevel.ORT_ENABLE_BASIC,
    "EXTENDED": ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED,
    "ALL": ort.GraphOptimizationLevel.ORT_ENABLE_ALL,
}

x_test = np.random.randn(32, 784).astype(np.float32)

print(f"{'Level':<12} {'Create(ms)':<12} {'Run(ms)':<12} {'Speedup':<10}")
print("-" * 48)

baseline_run = None
for name, level in levels.items():
    so = ort.SessionOptions()
    so.graph_optimization_level = level
    so.intra_op_num_threads = 4
    so.inter_op_num_threads = 1
    
    # Measure creation
    t0 = time.perf_counter()
    sess = ort.InferenceSession("session_demo.onnx", so, providers=["CPUExecutionProvider"])
    create_ms = (time.perf_counter() - t0) * 1000
    
    # Warmup
    for _ in range(20):
        sess.run(None, {"X": x_test})
    
    # Measure execution
    times = []
    for _ in range(200):
        t0 = time.perf_counter()
        sess.run(None, {"X": x_test})
        times.append((time.perf_counter() - t0) * 1000)
    
    run_ms = np.median(times)
    if baseline_run is None:
        baseline_run = run_ms
    speedup = baseline_run / run_ms
    
    print(f"{name:<12} {create_ms:<12.3f} {run_ms:<12.4f} {speedup:<10.2f}x")

<a id='3'></a>
## 3. RunOptions — Per-Invocation Controls

`RunOptions` provides per-request knobs that don't require rebuilding the session:

```
┌─────────────────────────────────────────────────────────────┐
│                       RunOptions                             │
├─────────────────────────────────────────────────────────────┤
│  terminate          │ Signal early termination               │
│  log_severity_level │ Override session log level for 1 run   │
│  log_verbosity_level│ Fine-grained verbosity                 │
│  tag                │ Annotation for profiling traces        │
│  add_run_config_entry│ EP-specific per-run config            │
└─────────────────────────────────────────────────────────────┘
```

### Termination Model

The `terminate` flag is cooperative — kernels check it periodically:

$$T_{\text{actual\_stop}} \leq T_{\text{terminate\_signal}} + T_{\text{kernel\_quantum}}$$

Where $T_{\text{kernel\_quantum}}$ is the maximum time between termination checks within a kernel (typically the time for one tile/block computation).

### Use Cases

| Scenario | RunOptions Usage |
|----------|------------------|
| Request timeout | Set terminate after SLA deadline |
| Debug single request | Increase log verbosity for one run |
| A/B trace comparison | Tag runs with experiment identifiers |
| Graceful shutdown | Terminate in-flight requests |

In [ ]:
import onnxruntime as ort
import numpy as np
import threading
import time

sess = ort.InferenceSession("session_demo.onnx", providers=["CPUExecutionProvider"])

# Demonstrate RunOptions
ro = ort.RunOptions()
ro.log_severity_level = 1  # Verbose for this run
ro.run_tag = "debug_request_42"

x = np.random.randn(1, 784).astype(np.float32)
result = sess.run(None, {"X": x}, run_options=ro)
print(f"Run with tag '{ro.run_tag}' completed, output shape: {result[0].shape}")

# Demonstrate termination (cooperative cancellation)
ro_cancel = ort.RunOptions()

def cancel_after(run_options, delay_sec):
    time.sleep(delay_sec)
    run_options.terminate = True
    print(f"  Termination signal sent after {delay_sec}s")

# For a large batch, cancellation could interrupt execution
large_x = np.random.randn(1024, 784).astype(np.float32)
t = threading.Thread(target=cancel_after, args=(ro_cancel, 0.001))
t.start()

try:
    result = sess.run(None, {"X": large_x}, run_options=ro_cancel)
    print(f"  Run completed before termination, shape: {result[0].shape}")
except ort.capi.onnxruntime_pybind11_state.RuntimeException as e:
    print(f"  Run terminated: {str(e)[:80]}")
t.join()

<a id='4'></a>
## 4. IOBinding — Zero-Copy GPU Inference

IOBinding is ORT's mechanism for **eliminating host-device memory transfers** in GPU inference pipelines. Without IOBinding, every `session.run()` call involves:

```
Standard session.run() path:
┌─────────┐     H2D Copy      ┌─────────────┐     D2H Copy     ┌──────────┐
│  NumPy  │───────────────────►│  GPU Kernel  │───────────────────►│  NumPy   │
│  Input  │   (PCIe transfer)  │  Execution   │   (PCIe transfer)  │  Output  │
└─────────┘                    └─────────────┘                    └──────────┘

IOBinding path:
┌───────────────┐              ┌─────────────┐              ┌────────────────┐
│  GPU Buffer   │─────────────►│  GPU Kernel  │─────────────►│  GPU Buffer    │
│  (pre-bound)  │  (no copy!)  │  Execution   │  (no copy!)  │  (pre-bound)   │
└───────────────┘              └─────────────┘              └────────────────┘
```

### Latency Model

The per-inference latency decomposes as:

$$\text{latency} = T_{\text{copy\_in}} + T_{\text{compute}} + T_{\text{copy\_out}} + T_{\text{sync}}$$

Without IOBinding:
$$T_{\text{copy\_in}} = \frac{|\text{input\_bytes}|}{\text{BW}_{\text{PCIe}}}, \quad T_{\text{copy\_out}} = \frac{|\text{output\_bytes}|}{\text{BW}_{\text{PCIe}}}$$

With IOBinding (data already on device):
$$T_{\text{copy\_in}} = 0, \quad T_{\text{copy\_out}} = 0$$

$$\text{Speedup}_{\text{IOBinding}} = \frac{T_{\text{copy\_in}} + T_{\text{compute}} + T_{\text{copy\_out}} + T_{\text{sync}}}{T_{\text{compute}} + T_{\text{sync}}}$$

For a model with 10ms compute and 2ms transfer overhead:
$$\text{Speedup} = \frac{2 + 10 + 2 + 0.01}{10 + 0.01} \approx 1.4\times$$

### Memory Ownership Model

IOBinding supports three memory management strategies:

| Strategy | Description | Use Case |
|----------|-------------|----------|
| ORT-allocated | ORT allocates output buffers on device | General GPU inference |
| User-allocated | User pre-allocates and binds output buffers | Pipeline with known output sizes |
| Shared memory | CUDA IPC or unified memory across processes | Multi-process serving |

### When to Use IOBinding

IOBinding is beneficial when:
1. Input data originates on GPU (e.g., from preprocessing pipeline)
2. Output feeds into another GPU operation (e.g., postprocessing)
3. The model is small enough that transfer time dominates compute
4. You're running inference in a tight loop with predictable I/O shapes

In [ ]:
import onnxruntime as ort
import numpy as np
import time

# Demonstrate IOBinding concepts (CPU example showing the API pattern)
# On GPU, this eliminates PCIe transfers
sess = ort.InferenceSession("session_demo.onnx", providers=["CPUExecutionProvider"])

# Standard run
x = np.random.randn(64, 784).astype(np.float32)
times_standard = []
for _ in range(20):
    sess.run(None, {"X": x})
for _ in range(500):
    t0 = time.perf_counter()
    sess.run(None, {"X": x})
    times_standard.append((time.perf_counter() - t0) * 1000)

# IOBinding run (on CPU, demonstrates API; real benefit is on GPU)
io_binding = sess.io_binding()
x_ortvalue = ort.OrtValue.ortvalue_from_numpy(x, "cpu", 0)

times_iobinding = []
for _ in range(20):
    io_binding.bind_ortvalue_input("X", x_ortvalue)
    io_binding.bind_output("Y", "cpu", 0)
    sess.run_with_iobinding(io_binding)

for _ in range(500):
    io_binding.bind_ortvalue_input("X", x_ortvalue)
    io_binding.bind_output("Y", "cpu", 0)
    t0 = time.perf_counter()
    sess.run_with_iobinding(io_binding)
    times_iobinding.append((time.perf_counter() - t0) * 1000)

print("Latency comparison (batch=64):")
print(f"  Standard run:  mean={np.mean(times_standard):.4f}ms  p50={np.median(times_standard):.4f}ms")
print(f"  IOBinding run: mean={np.mean(times_iobinding):.4f}ms  p50={np.median(times_iobinding):.4f}ms")
print(f"\nNote: On CPU, IOBinding shows minimal benefit.")
print(f"On GPU, it eliminates PCIe H2D/D2H transfers for significant speedup.")

<a id='5'></a>
## 5. Input/Output Management

### OrtValue — The Universal Tensor Container

`OrtValue` wraps tensor data with metadata about where it lives:

```
┌─────────────────────────────────────────────┐
│                  OrtValue                    │
├─────────────────────────────────────────────┤
│  data_ptr        → raw buffer pointer       │
│  shape           → [dim0, dim1, ...]        │
│  dtype           → FLOAT, INT8, FLOAT16...  │
│  device_type     → cpu, cuda, dml...        │
│  device_id       → 0, 1, 2...              │
│  is_tensor       → True/False               │
│  owns_data       → True/False               │
└─────────────────────────────────────────────┘
```

### Feed Dictionary Semantics

When you pass `{"input_name": numpy_array}` to `session.run()`:

1. ORT wraps the numpy array in an `OrtValue` (zero-copy on CPU, since numpy is contiguous)
2. If the EP requires device memory, ORT copies the data (H2D transfer)
3. Kernels execute using the device-local buffer
4. Output is either returned as numpy (D2H copy) or kept as `OrtValue`

### Type Matching Rules

The input tensor must match the model's expected type:

$$\text{feed}[\text{name}].\text{dtype} = \text{model\_input}[\text{name}].\text{elem\_type}$$

Common type mismatches:

| Python Default | ONNX Expected | Fix |
|---------------|---------------|---------|
| `float64` | `FLOAT` (fp32) | `.astype(np.float32)` |
| `int64` | `INT32` | `.astype(np.int32)` |
| Python list | Contiguous array | `np.array(...)` |

In [ ]:
import onnxruntime as ort
import numpy as np

sess = ort.InferenceSession("session_demo.onnx", providers=["CPUExecutionProvider"])

# Inspect model metadata
print("=" * 60)
print("MODEL I/O METADATA")
print("=" * 60)

print("\nInputs:")
for inp in sess.get_inputs():
    print(f"  Name: {inp.name}")
    print(f"  Type: {inp.type}")
    print(f"  Shape: {inp.shape}")

print("\nOutputs:")
for out in sess.get_outputs():
    print(f"  Name: {out.name}")
    print(f"  Type: {out.type}")
    print(f"  Shape: {out.shape}")

# Demonstrate OrtValue creation and inspection
print("\n" + "=" * 60)
print("OrtValue INSPECTION")
print("=" * 60)

x = np.random.randn(8, 784).astype(np.float32)
ortval = ort.OrtValue.ortvalue_from_numpy(x, "cpu", 0)

print(f"\n  Shape: {ortval.shape()}")
print(f"  Data type: {ortval.data_type()}")
print(f"  Device: {ortval.device_name()}")
print(f"  Is tensor: {ortval.is_tensor()}")

# Run with output names selection
result = sess.run(["Y"], {"X": x})
print(f"\nOutput 'Y' shape: {result[0].shape}")
print(f"Output 'Y' dtype: {result[0].dtype}")
print(f"Output sums to ~1.0 per row (softmax): {result[0].sum(axis=1)[:3]}")

<a id='6'></a>
## 6. Dynamic Shapes — Symbolic Dimensions

ONNX models can have **symbolic dimensions** — dimensions whose size varies at runtime:

```
Static shape:   [1, 3, 224, 224]    — fixed at export
Dynamic shape:  ["batch", 3, "H", "W"]  — resolved at runtime
Mixed:          ["batch", 3, 224, 224]  — batch varies, spatial fixed
```

### Shape Resolution Pipeline

```
Export time:     shape = ["batch", 784]     (symbolic)
                        │
Session creation:       │  (no resolution yet)
                        │
run() call:      input shape = [32, 784]    (concrete)
                        │
Kernel dispatch:  uses concrete shape for buffer allocation
```

### Shape Algebra

For a MatMul node computing $C = A \times B$:

$$A: [\text{batch}, M, K] \times B: [K, N] \to C: [\text{batch}, M, N]$$

Shape constraints propagate symbolically:
- If input is $[\text{batch}, 784]$ and weight is $[784, 256]$
- Output is $[\text{batch}, 256]$ — `batch` remains symbolic

### Performance Implications

Dynamic shapes can impact performance:

| Aspect | Static Shapes | Dynamic Shapes |
|--------|--------------|----------------|
| Memory planning | Optimal (exact allocation) | May over-allocate |
| Kernel selection | Best kernel for exact size | Generic kernel |
| TensorRT | Single engine | Multiple profiles needed |
| Memory pattern | Perfect reuse | Pattern reuse if shapes repeat |

### Shape Bucketing Strategy

For production systems with varying input sizes, bucket inputs into a small set of shapes:

$$\text{bucket}(n) = \min\{b \in \text{buckets} : b \geq n\}$$

Example: buckets = {1, 4, 8, 16, 32, 64, 128}

This ensures ORT's memory pattern is reused, avoiding allocator churn.

In [ ]:
import onnxruntime as ort
import numpy as np
import time

sess = ort.InferenceSession("session_demo.onnx", providers=["CPUExecutionProvider"])

# Our model has dynamic "batch" dimension
print("Input shape from model:", sess.get_inputs()[0].shape)
print("  'batch' is symbolic — any batch size works at runtime\n")

# Test various batch sizes
batch_sizes = [1, 4, 8, 16, 32, 64, 128, 256]
print(f"{'Batch':<8} {'Latency(ms)':<14} {'Throughput(s/s)':<18} {'us/sample':<12}")
print("-" * 54)

for bs in batch_sizes:
    x = np.random.randn(bs, 784).astype(np.float32)
    
    # Warmup
    for _ in range(10):
        sess.run(None, {"X": x})
    
    # Measure
    times = []
    for _ in range(100):
        t0 = time.perf_counter()
        sess.run(None, {"X": x})
        times.append((time.perf_counter() - t0) * 1000)
    
    lat = np.median(times)
    tput = bs / (lat / 1000)
    per_sample = lat * 1000 / bs  # microseconds
    print(f"{bs:<8} {lat:<14.4f} {tput:<18.0f} {per_sample:<12.2f}")

<a id='7'></a>
## 7. Batching Strategies

### Strategy Comparison

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    Batching Strategies                                    │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                          │
│  1. STATIC BATCHING                                                      │
│     Always process exactly N samples                                     │
│     ┌────┬────┬────┬────┐                                               │
│     │ s1 │ s2 │ s3 │ s4 │ → run(batch=4) → 4 results                   │
│     └────┴────┴────┴────┘                                               │
│     + Simple, predictable memory  - Blocks until batch full              │
│                                                                          │
│  2. DYNAMIC BATCHING                                                     │
│     Accumulate up to N, fire on timeout or full                          │
│     ┌────┬────┬─  ─┐                                                    │
│     │ s1 │ s2 │ .. │ → timeout → run(batch=2) → pad/unpad               │
│     └────┴────┴─  ─┘                                                    │
│     + Reduces tail latency  - More complex orchestration                 │
│                                                                          │
│  3. CONTINUOUS BATCHING (for generative models)                          │
│     New requests join mid-generation                                     │
│     Step t: [s1, s2, s3]    Step t+1: [s1, s2, s3, s4_new]             │
│     + Maximizes GPU utilization  - Complex state management              │
│                                                                          │
└─────────────────────────────────────────────────────────────────────────┘
```

### Throughput vs Latency Tradeoff

Larger batches improve throughput but increase per-sample latency:

$$\text{Throughput}(B) = \frac{B}{T(B)}$$

$$\text{Per-sample latency}(B) = \frac{T(B)}{B} + T_{\text{queue\_wait}}$$

Where $T(B)$ is the execution time for batch size $B$. Due to hardware parallelism:

$$T(B) = T(1) + \alpha \cdot (B - 1), \quad \alpha < 1$$

The per-sample latency initially decreases (amortizing fixed overhead) then increases (compute saturation):

$$B^* = \argmin_B \frac{T(B)}{B} \quad \text{(optimal batch size)}$$

### Padding Efficiency

When actual batch size doesn't match the bucket:

$$\text{Compute efficiency} = \frac{B_{\text{actual}}}{B_{\text{padded}}}$$

With buckets $\{1, 4, 8, 16, 32\}$ and actual batch of 5:
$$\text{efficiency} = \frac{5}{8} = 62.5\%$$

In [ ]:
import numpy as np
import time
import onnxruntime as ort
import matplotlib.pyplot as plt

sess = ort.InferenceSession("session_demo.onnx", providers=["CPUExecutionProvider"])

# Measure throughput vs batch size
batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
latencies = []
throughputs = []
per_sample = []

for bs in batch_sizes:
    x = np.random.randn(bs, 784).astype(np.float32)
    for _ in range(10):
        sess.run(None, {"X": x})
    
    times = []
    for _ in range(100):
        t0 = time.perf_counter()
        sess.run(None, {"X": x})
        times.append((time.perf_counter() - t0) * 1000)
    
    med = np.median(times)
    latencies.append(med)
    throughputs.append(bs / (med / 1000))
    per_sample.append(med / bs)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(batch_sizes, latencies, 'bo-', linewidth=2)
axes[0].set_xlabel('Batch Size')
axes[0].set_ylabel('Latency (ms)')
axes[0].set_title('Total Latency vs Batch Size')
axes[0].set_xscale('log', base=2)
axes[0].grid(True, alpha=0.3)

axes[1].plot(batch_sizes, throughputs, 'go-', linewidth=2)
axes[1].set_xlabel('Batch Size')
axes[1].set_ylabel('Throughput (samples/sec)')
axes[1].set_title('Throughput vs Batch Size')
axes[1].set_xscale('log', base=2)
axes[1].grid(True, alpha=0.3)

axes[2].plot(batch_sizes, [p*1000 for p in per_sample], 'ro-', linewidth=2)
axes[2].set_xlabel('Batch Size')
axes[2].set_ylabel('Per-sample Latency (μs)')
axes[2].set_title('Amortized Per-sample Cost')
axes[2].set_xscale('log', base=2)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('session_batching.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nOptimal batch size for throughput: {batch_sizes[np.argmax(throughputs)]}")
print(f"Peak throughput: {max(throughputs):.0f} samples/sec")

<a id='8'></a>
## 8. Latency Decomposition

Understanding where time is spent during `session.run()` is critical for optimization. The total latency decomposes into:

$$T_{\text{total}} = T_{\text{copy\_in}} + T_{\text{compute}} + T_{\text{copy\_out}} + T_{\text{sync}} + T_{\text{overhead}}$$

### Component Breakdown

```
Time ────────────────────────────────────────────────────────────────────►
                                                                          
│◄── T_copy_in ──►│◄────── T_compute ──────►│◄── T_copy_out ──►│◄─T_sync─►│
│                  │                          │                   │          │
│  Copy numpy to   │  Execute kernel graph    │  Copy results to  │  Stream  │
│  device memory   │  (GPU/CPU kernels)       │  host memory      │  sync    │
│                  │                          │                   │          │
│  Proportional    │  Proportional to         │  Proportional     │  Fixed   │
│  to input size   │  model FLOPs             │  to output size   │  cost    │
```

### Formal Model

For a model with $F$ FLOPs, input size $S_{in}$ bytes, output size $S_{out}$ bytes:

$$T_{\text{copy\_in}} = \frac{S_{in}}{\text{BW}_{\text{bus}}}$$

$$T_{\text{compute}} = \max\left(\frac{F}{\text{FLOPS}_{\text{peak}}}, \frac{S_{\text{activations}}}{\text{BW}_{\text{memory}}}\right)$$

$$T_{\text{copy\_out}} = \frac{S_{out}}{\text{BW}_{\text{bus}}}$$

The compute time follows the **roofline model** — it's bounded by either compute or memory bandwidth, whichever is the bottleneck.

### Profiling Methodology

ORT's built-in profiler breaks down execution into:
1. **Session initialization** (one-time)
2. **Sequential execution** (per-run, per-node timing)
3. **Fence/sync operations** (per-EP-boundary)

The profile outputs a Chrome trace file (`.json`) viewable in `chrome://tracing`.

In [ ]:
import onnxruntime as ort
import numpy as np
import time
import json

# Profile a session to decompose latency
so = ort.SessionOptions()
so.enable_profiling = True
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

sess = ort.InferenceSession("session_demo.onnx", so, providers=["CPUExecutionProvider"])

# Run profiled inference
x = np.random.randn(32, 784).astype(np.float32)
for _ in range(5):
    sess.run(None, {"X": x})

# Get profile
prof_file = sess.end_profiling()
print(f"Profile saved to: {prof_file}")

# Parse and summarize
with open(prof_file, 'r') as f:
    profile_data = json.load(f)

# Aggregate by category
kernel_times = {}
for event in profile_data:
    if 'dur' in event and 'name' in event:
        name = event.get('name', 'unknown')
        dur = event['dur']  # microseconds
        cat = event.get('cat', 'other')
        if cat == 'Node':
            kernel_times[name] = kernel_times.get(name, 0) + dur

if kernel_times:
    print(f"\nKernel timing breakdown (total across all runs):")
    print(f"{'Kernel':<30} {'Time (μs)':<12} {'%':<8}")
    print("-" * 52)
    total = sum(kernel_times.values())
    for name, dur in sorted(kernel_times.items(), key=lambda x: -x[1])[:10]:
        print(f"{name:<30} {dur:<12.1f} {dur/total*100:<8.1f}")
    print(f"{'TOTAL':<30} {total:<12.1f} {'100.0':<8}")
else:
    print("\nProfile data structure varies by ORT version.")
    print(f"Events found: {len(profile_data)}")

# Clean up profile file
import os
os.remove(prof_file)

<a id='9'></a>
## 9. Session Caching and Warmup Patterns

### Why Warmup Matters

The first few inference calls exhibit higher latency due to:
1. **Memory pattern recording** — ORT observes allocation patterns on first run
2. **JIT compilation** — Some EPs compile kernels on first invocation
3. **Cache warming** — CPU caches are cold for model weights
4. **Thread pool spin-up** — Worker threads may not be active yet

```
Latency
  │
  │ ┌─┐
  │ │ │ ┌─┐
  │ │ │ │ │ ┌─┐
  │ │ │ │ │ │ │ ┌───────────────────────── steady state
  │ │ │ │ │ │ │ │
  └─┴─┴─┴─┴─┴─┴─┴──────────────────────── Run #
    1   2   3   4   5   6   7   8 ...
    └───── warmup period ────┘
```

### Production Session Management

```
┌─────────────────────────────────────────────────────────────────┐
│              Production Deployment Pattern                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│  Server Startup:                                                 │
│    1. Load model from disk/blob storage                          │
│    2. Create InferenceSession (expensive)                        │
│    3. Run N warmup inferences with representative shapes         │
│    4. Mark server as ready (health check passes)                 │
│                                                                  │
│  Request Handling:                                               │
│    5. Reuse session for all requests (thread-safe!)              │
│    6. No per-request session creation                            │
│                                                                  │
│  Model Update:                                                   │
│    7. Create new session in background                           │
│    8. Warm up new session                                        │
│    9. Atomic swap (pointer replacement)                          │
│   10. Old session garbage collected                              │
│                                                                  │
└─────────────────────────────────────────────────────────────────┘
```

### Thread Safety

A single `InferenceSession` can serve multiple concurrent requests:

$$\text{Concurrent capacity} = \frac{N_{\text{cores}}}{\text{intra\_op\_threads}}$$

With 32 cores and 4 intra-op threads, you can serve 8 concurrent requests without contention.

In [ ]:
import onnxruntime as ort
import numpy as np
import time
import matplotlib.pyplot as plt

# Demonstrate warmup behavior
so = ort.SessionOptions()
so.enable_mem_pattern = True
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

sess = ort.InferenceSession("session_demo.onnx", so, providers=["CPUExecutionProvider"])

x = np.random.randn(32, 784).astype(np.float32)

# Measure each run individually to see warmup
all_times = []
for i in range(100):
    t0 = time.perf_counter()
    sess.run(None, {"X": x})
    all_times.append((time.perf_counter() - t0) * 1000)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Plot all runs showing warmup
ax1.plot(range(1, 101), all_times, 'b-', linewidth=1, alpha=0.7)
ax1.axhline(y=np.median(all_times[10:]), color='r', linestyle='--', label=f'Steady state median: {np.median(all_times[10:]):.3f}ms')
ax1.axvspan(1, 5, alpha=0.2, color='orange', label='Warmup region')
ax1.set_xlabel('Run #', fontsize=11)
ax1.set_ylabel('Latency (ms)', fontsize=11)
ax1.set_title('Inference Latency: Warmup → Steady State', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Latency distribution (excluding warmup)
steady_times = all_times[10:]
ax2.hist(steady_times, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
ax2.axvline(x=np.median(steady_times), color='r', linestyle='--', label=f'Median: {np.median(steady_times):.4f}ms')
ax2.axvline(x=np.percentile(steady_times, 99), color='orange', linestyle='--', label=f'P99: {np.percentile(steady_times, 99):.4f}ms')
ax2.set_xlabel('Latency (ms)', fontsize=11)
ax2.set_ylabel('Count', fontsize=11)
ax2.set_title('Steady-State Latency Distribution', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('session_warmup.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFirst run:  {all_times[0]:.4f} ms")
print(f"Steady P50: {np.median(steady_times):.4f} ms")
print(f"Warmup penalty: {all_times[0] / np.median(steady_times):.1f}x")

<a id='10'></a>
## 10. Error Handling and Failure Modes

### Error Categories

```
┌────────────────────────────────────────────────────────────────────────┐
│                    ORT Error Taxonomy                                   │
├────────────────────────────────────────────────────────────────────────┤
│                                                                        │
│  SESSION CREATION ERRORS (fatal, non-recoverable for this model)       │
│  ├── Model loading failure (corrupt file, missing external data)       │
│  ├── Unsupported op/version (NOT_IMPLEMENTED)                          │
│  ├── Invalid model (fails onnx.checker)                                │
│  └── EP initialization failure (missing CUDA, wrong driver)            │
│                                                                        │
│  RUNTIME ERRORS (per-request, potentially recoverable)                 │
│  ├── Shape mismatch (input doesn't match expected dimensions)          │
│  ├── Type mismatch (float64 when float32 expected)                     │
│  ├── Missing input (feed dict missing required key)                    │
│  ├── OOM (out of memory on device)                                     │
│  └── Numerical errors (NaN/Inf propagation)                            │
│                                                                        │
│  PERFORMANCE DEGRADATION (silent, no exception)                        │
│  ├── EP fallback (preferred EP can't handle some ops)                  │
│  ├── Suboptimal partitioning (too many EP boundaries)                  │
│  └── Allocator churn (varying shapes without bucketing)                │
│                                                                        │
└────────────────────────────────────────────────────────────────────────┘
```

### Defensive Inference Pattern

A production-grade inference wrapper should:

1. **Validate inputs** before calling run (shape, dtype, range)
2. **Catch ORT exceptions** and classify as retryable vs fatal
3. **Monitor for silent degradation** (latency regression, EP fallback)
4. **Log context** for debugging (input metadata, session config)

In [ ]:
import onnxruntime as ort
import numpy as np

sess = ort.InferenceSession("session_demo.onnx", providers=["CPUExecutionProvider"])

class InferenceError(Exception):
    """Base class for inference errors."""
    pass

class InputValidationError(InferenceError):
    pass

class RuntimeInferenceError(InferenceError):
    pass

def safe_inference(session, inputs: dict) -> list:
    """Production-grade inference with validation and error handling."""
    # 1. Validate inputs
    expected_inputs = {inp.name: inp for inp in session.get_inputs()}
    
    for name, meta in expected_inputs.items():
        if name not in inputs:
            raise InputValidationError(f"Missing input: '{name}'")
        
        arr = inputs[name]
        if not isinstance(arr, np.ndarray):
            raise InputValidationError(f"Input '{name}' must be numpy array, got {type(arr)}")
        
        # Check dtype
        expected_type = meta.type
        if 'float' in expected_type and arr.dtype != np.float32:
            raise InputValidationError(
                f"Input '{name}' dtype mismatch: expected float32, got {arr.dtype}")
        
        # Check rank
        if len(arr.shape) != len(meta.shape):
            raise InputValidationError(
                f"Input '{name}' rank mismatch: expected {len(meta.shape)}, got {len(arr.shape)}")
    
    # 2. Execute with error handling
    try:
        return session.run(None, inputs)
    except Exception as e:
        raise RuntimeInferenceError(f"Inference failed: {e}") from e

# Test valid input
x_valid = np.random.randn(4, 784).astype(np.float32)
result = safe_inference(sess, {"X": x_valid})
print(f"Valid inference: output shape = {result[0].shape}")

# Test error cases
test_cases = [
    ({}, "Missing input"),
    ({"X": np.random.randn(4, 784)}, "Wrong dtype (float64)"),
    ({"X": np.random.randn(784).astype(np.float32)}, "Wrong rank (1D instead of 2D)"),
]

print("\nError handling tests:")
for feed, desc in test_cases:
    try:
        safe_inference(sess, feed)
    except InferenceError as e:
        print(f"  [{desc}] Caught: {type(e).__name__}: {e}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Comprehensive session performance visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Latency decomposition (conceptual for GPU scenario)
components = ['Copy\nH→D', 'Compute', 'Copy\nD→H', 'Sync', 'Overhead']
# Without IOBinding
without_iob = [2.0, 8.0, 1.5, 0.3, 0.2]
# With IOBinding
with_iob = [0.0, 8.0, 0.0, 0.1, 0.1]

x = np.arange(len(components))
width = 0.35
axes[0,0].bar(x - width/2, without_iob, width, label='Standard run', color='#e74c3c')
axes[0,0].bar(x + width/2, with_iob, width, label='With IOBinding', color='#2ecc71')
axes[0,0].set_xlabel('Component')
axes[0,0].set_ylabel('Time (ms)')
axes[0,0].set_title('Latency Decomposition\n(GPU scenario, conceptual)', fontweight='bold')
axes[0,0].set_xticks(x)
axes[0,0].set_xticklabels(components)
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3, axis='y')

# Plot 2: Session creation vs N inferences (amortization)
N_values = np.logspace(0, 6, 50)
T_create = 50  # ms
T_run = 0.5    # ms
amortized = T_create / N_values + T_run

axes[0,1].semilogx(N_values, amortized, 'b-', linewidth=2)
axes[0,1].axhline(y=T_run, color='r', linestyle='--', label=f'Asymptote: T_run = {T_run}ms')
axes[0,1].axhline(y=T_create + T_run, color='gray', linestyle=':', alpha=0.5)
axes[0,1].set_xlabel('Number of Inference Calls (N)')
axes[0,1].set_ylabel('Amortized Cost per Call (ms)')
axes[0,1].set_title('Session Creation Cost Amortization\n$C_{amortized} = T_{create}/N + T_{run}$', fontweight='bold')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Plot 3: Thread scaling efficiency
threads = [1, 2, 4, 8, 16]
p_values = [0.7, 0.85, 0.95]
for p in p_values:
    speedups = [1/((1-p) + p/t) for t in threads]
    efficiency = [s/t * 100 for s, t in zip(speedups, threads)]
    axes[1,0].plot(threads, efficiency, 'o-', linewidth=2, label=f'p={p}')

axes[1,0].set_xlabel('Number of Threads')
axes[1,0].set_ylabel('Efficiency (%)')
axes[1,0].set_title('Threading Efficiency\n$E(T) = S(T)/T \\times 100$', fontweight='bold')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)
axes[1,0].set_ylim([0, 105])

# Plot 4: Shape bucketing efficiency
actual_batches = range(1, 65)
buckets = [1, 4, 8, 16, 32, 64]
efficiencies = []
for b in actual_batches:
    bucket = min(bk for bk in buckets if bk >= b)
    efficiencies.append(b / bucket * 100)

axes[1,1].plot(list(actual_batches), efficiencies, 'b-', linewidth=1.5)
axes[1,1].fill_between(list(actual_batches), efficiencies, alpha=0.2)
for bk in buckets:
    axes[1,1].axvline(x=bk, color='red', linestyle='--', alpha=0.3)

axes[1,1].set_xlabel('Actual Batch Size')
axes[1,1].set_ylabel('Compute Efficiency (%)')
axes[1,1].set_title('Shape Bucketing Efficiency\nBuckets: {1, 4, 8, 16, 32, 64}', fontweight='bold')
axes[1,1].grid(True, alpha=0.3)
axes[1,1].set_ylim([0, 105])

plt.tight_layout()
plt.savefig('session_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Key Equations Summary

### Latency Model

$$\text{latency} = T_{\text{copy}} + T_{\text{compute}} + T_{\text{sync}}$$

$$T_{\text{copy}} = \frac{S_{\text{in}} + S_{\text{out}}}{\text{BW}_{\text{PCIe}}}$$

### Amortized Session Cost

$$C_{\text{amortized}}(N) = \frac{T_{\text{create}}}{N} + T_{\text{run}}$$

### Threading Efficiency

$$S(T) = \frac{1}{(1-p) + p/T}, \quad E(T) = \frac{S(T)}{T}$$

### Throughput

$$\text{Throughput} = \frac{B}{T(B)} \quad [\text{samples/sec}]$$

### Bucketing Efficiency

$$\eta = \frac{B_{\text{actual}}}{\text{bucket}(B_{\text{actual}})}$$

### Concurrent Capacity

$$\text{max\_concurrent} = \left\lfloor \frac{N_{\text{cores}}}{\text{intra\_op\_threads}} \right\rfloor$$

In [ ]:
# Cleanup
import os
for f in ['session_demo.onnx']:
    if os.path.exists(f):
        os.remove(f)
# Remove any generated plots
for f in ['session_batching.png', 'session_warmup.png', 'session_analysis.png']:
    if os.path.exists(f):
        os.remove(f)
print("Cleanup complete.")

## Summary

The `InferenceSession` is ORT's compiled execution artifact:

1. **Session creation** is expensive (graph optimization, EP partitioning, memory planning) but amortized across millions of runs

2. **SessionOptions** controls the optimization-latency tradeoff at creation time — higher levels invest more upfront for faster steady-state execution

3. **IOBinding** eliminates host-device transfers for GPU inference, reducing latency by $\frac{T_{\text{copy}}}{T_{\text{total}}}$

4. **Dynamic shapes** provide flexibility but require bucketing strategies to maintain memory pattern reuse

5. **Batching** trades latency for throughput — optimal batch size maximizes hardware utilization without exceeding memory

6. **Warmup** is essential in production to stabilize memory patterns and cache state before accepting traffic

7. **Thread safety** allows a single session to serve concurrent requests, with capacity bounded by core count / intra-op threads